# Feature cache build (S6)

Builds the two feature caches for AcousticWise on Kaggle: every clip of the UniqueShip *12 Class - 5 Hours Each* bundle goes through the package's `load_audio` → `make_windows` → `log_mel` once, at 32 kHz and at 16 kHz.

**Inputs:** the `uniqueship-12-class-5-hours` dataset and the private `acousticwise-src` dataset (the `src/acousticwise` package, `scripts/cache_features.py` and `configs/default.yaml`, uploaded with `notebooks/spectrogram_grid/upload_src.sh`). CPU only, no internet.

**Output** in `/kaggle/working/cache/`: `sr32000/` and `sr16000/`, each with `logmel.npy`, `index.csv` and `meta.json` (technical design, section 5), plus `checksums.txt` to verify the download.

## 1. The package and the script

In [ ]:
import glob
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

src_hits = glob.glob("/kaggle/input/**/acousticwise/__init__.py", recursive=True)
assert src_hits, "Attach the acousticwise-src dataset (Add Input)."
SRC = Path(src_hits[0]).parent.parent
SCRIPT = next(iter(glob.glob("/kaggle/input/**/scripts/cache_features.py", recursive=True)), None)
CONFIG = next(iter(glob.glob("/kaggle/input/**/configs/default.yaml", recursive=True)), None)
assert SCRIPT and CONFIG, "acousticwise-src is missing scripts/ or configs/: rerun upload_src.sh."
OUT = Path("/kaggle/working/cache")
print("package:", SRC)
print("script: ", SCRIPT)
print("config: ", CONFIG)

## 2. Extract the bundle

Joins the split zip into `/tmp/joined.zip`, unzips it to `/tmp/stage` (about 26 GB) and deletes the joined zip, as in `kaggle_data_prep.ipynb`. About half an hour.

In [ ]:
STAGE = Path("/tmp/stage")
if not (STAGE / "DONE").exists():
    hits = glob.glob("/kaggle/input/**/allclass_5hours.z01", recursive=True)
    assert hits, "Attach the uniqueship-12-class-5-hours dataset (Add Input)."
    D = os.path.dirname(hits[0])
    assert os.path.exists(f"{D}/allclass_5hours.zip.part"), "Use a dataset version with both zip parts."
    os.makedirs("/tmp/parts", exist_ok=True)
    for src, dst in [("allclass_5hours.z01", "allclass_5hours.z01"),
                     ("allclass_5hours.zip.part", "allclass_5hours.zip")]:
        if not os.path.lexists(f"/tmp/parts/{dst}"):
            os.symlink(f"{D}/{src}", f"/tmp/parts/{dst}")
    t = time.time()
    subprocess.run("cd /tmp && rm -rf stage joined.zip && zip -q -s 0 parts/allclass_5hours.zip --out joined.zip"
                   " && unzip -q joined.zip -d stage && rm joined.zip && touch stage/DONE",
                   shell=True, check=True)
    print(f"extracted in {time.time() - t:.0f} s")

fold_dirs = glob.glob(str(STAGE / "**/fold_1"), recursive=True)
assert len(fold_dirs) == 1, fold_dirs
BUNDLE = Path(fold_dirs[0]).parent
print("bundle:", BUNDLE)
print(subprocess.run(["du", "-sh", str(STAGE)], capture_output=True, text=True).stdout)

## 3. Build both caches

Runs `scripts/cache_features.py` with the package on the path. The script reads every clip from fold 1's train, validation and test CSVs (together, every clip once) and builds `sr32000/` then `sr16000/`. The build stops if two clips share an id or more than 0.1% fail to load.

In [ ]:
t = time.time()
subprocess.run(
    [sys.executable, SCRIPT, "--config", CONFIG, "--bundle", str(BUNDLE), "--out", str(OUT),
     "--sample-rates", "32000", "16000"],
    env={**os.environ, "PYTHONPATH": str(SRC)},
    check=True,
)
print(f"both caches built in {time.time() - t:.0f} s")

## 4. Check the caches

For each cache: the array shape and type, the clips per class, the padded-clip share (should be under about 1%), the failed clips with their reasons, and a SHA-256 of every file, saved to `checksums.txt` to compare after the download.

In [ ]:
import numpy as np
import pandas as pd

lines = []
for sr_dir in sorted(OUT.glob("sr*")):
    logmel = np.load(sr_dir / "logmel.npy", mmap_mode="r")
    index = pd.read_csv(sr_dir / "index.csv")
    meta = json.loads((sr_dir / "meta.json").read_text())
    n = len(index)
    print(f"== {sr_dir.name}: logmel {logmel.shape} {logmel.dtype}, index {n} rows")
    assert logmel.shape[0] == n and list(index.row) == list(range(n))
    print("settings:", meta["settings"])
    print(f"padded: {meta['padded_count']} of {n} = {100 * meta['padded_count'] / n:.2f}%")
    print(f"failed: {len(meta['failed'])}")
    for f in meta["failed"][:20]:
        print("  ", f["clip_id"], "-", f["reason"])
    print("clips per class:", index.label.value_counts().sort_index().to_dict())
    print(f"peak_db {index.peak_db.min():.1f} to {index.peak_db.max():.1f}, "
          f"rms_db {index.rms_db.min():.1f} to {index.rms_db.max():.1f}")
    for path in sorted(sr_dir.iterdir()):
        h = hashlib.sha256()
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 24), b""):
                h.update(chunk)
        lines.append(f"{h.hexdigest()}  {sr_dir.name}/{path.name}")
        print(f"  {path.name}: {path.stat().st_size / 1e9:.2f} GB")
    print()

(OUT / "checksums.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## After the run

1. On kaggle.com/datasets, **New Dataset → Notebook output**, pick this notebook: save `cache/` as the private dataset `acousticwise-cache`, so later Kaggle runs can attach it without extracting the audio.
2. On the laptop: `kaggle kernels output yanyetang/acousticwise-cache-build -p data/` then, inside `data/cache/`, `shasum -a 256 -c checksums.txt`.